# Multi-Layer QK Spectral Segmentation

**Improvements over the baseline deep-spectral-segmentation:**

| | Baseline | This method |
|---|---|---|
| Features | K only, last layer | Q+K, layers 3/6/9/12 |
| Affinity info | Appearance similarity | Multi-scale + semantic attention |
| Eigensolver | `eigsh` on N×N Laplacian | Randomized SVD on N×D features |
| Color affinity | Required (λ hand-tuned) | Not needed |
| VOC2012 runtime | ~10 days / 100 CPUs | ~5 min / 2×T4 |

**Benchmarks supported:** Object Localization · Object Segmentation · Semantic Segmentation

## 0. Install dependencies

In [ ]:
!pip install -q fire accelerate scikit-learn tqdm pymatting
# SimpleCRF is optional (only needed for CRF post-processing)
# !pip install -q SimpleCRF

## 1. Clone repo and set up paths

In [ ]:
import os

REPO = '/kaggle/working/deep-spectral-segmentation'

if not os.path.exists(REPO):
    !git clone https://github.com/welu2027/qk-spectral-segmentation.git {REPO}

os.chdir(REPO)
print('Working directory:', os.getcwd())

In [ ]:
# Patch: force reload of extract_multilayer_qk after git clone
import importlib, sys, shutil, os

# Clear any stale partial files from previous runs
for d in [EIGS_DIR]:
    if os.path.isdir(d):
        shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)
        print(f'Cleared {d}')

if 'extract_multilayer_qk' in sys.modules:
    importlib.reload(sys.modules['extract_multilayer_qk'])
    print('Module reloaded')
print('Ready')

## 2. Set up VOC2012 dataset

**Option A** — Use the Kaggle VOC2012 dataset (add it via the Datasets panel on the right).

**Option B** — Download directly (slower but no extra steps).

In [ ]:
import os
from pathlib import Path

# Option A: vijayabhaskar96/pascal-voc-2007-and-2012
KAGGLE_VOC = Path('/kaggle/input/datasets/vijayabhaskar96/pascal-voc-2007-and-2012/VOCdevkit/VOC2012')

if KAGGLE_VOC.exists() and list(KAGGLE_VOC.glob('JPEGImages/*.jpg')):
    print(f'Using clean VOC2012 from vijayabhaskar96 dataset: {KAGGLE_VOC}')
else:
    # Option B: Fallback — download official tar (~2 GB, takes ~5 min)
    print('Clean dataset not found — downloading official VOC2012 tar...')
    os.makedirs('/kaggle/working/voc', exist_ok=True)
    os.system('wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar -O /kaggle/working/voc/voc2012.tar')
    os.system('tar -xf /kaggle/working/voc/voc2012.tar -C /kaggle/working/voc/')
    KAGGLE_VOC = Path('/kaggle/working/voc/VOCdevkit/VOC2012')
    print(f'Downloaded VOC2012 to: {KAGGLE_VOC}')

KAGGLE_VOC    = str(KAGGLE_VOC)
IMAGES_ROOT   = f'{KAGGLE_VOC}/JPEGImages'
ANNO_ROOT     = f'{KAGGLE_VOC}/Annotations'
SEG_ROOT      = f'{KAGGLE_VOC}/SegmentationClass'
SEG_OBJ_ROOT  = f'{KAGGLE_VOC}/SegmentationObject'

DATA_DIR = f'{REPO}/data/VOC2012'
os.makedirs(f'{DATA_DIR}/lists', exist_ok=True)

# Build image list (always rebuild so it stays fresh)
images_list_path = f'{DATA_DIR}/lists/images.txt'
imgs = sorted(Path(IMAGES_ROOT).glob('*.jpg'))
assert len(imgs) > 0, f'No .jpg files found in {IMAGES_ROOT} — check dataset path'
with open(images_list_path, 'w') as f:
    f.write('\n'.join(img.name for img in imgs))
print(f'Written {len(imgs)} image filenames to {images_list_path}')
print(f'Sample: {imgs[0].name} ✓')

## 3. Configure paths

In [ ]:
import sys
sys.path.insert(0, f'{REPO}/extract')

MODEL_NAME   = 'dino_vits16'
BATCH_SIZE   = 1
K            = 20       # number of eigenvectors
NUM_SEGMENTS = 4        # segments per image (non-adaptive)

FEATURES_DIR  = f'{DATA_DIR}/features/{MODEL_NAME}_multilayer'
EIGS_DIR      = f'{DATA_DIR}/eigs/multilayer_svd'
SEGS_DIR      = f'{DATA_DIR}/segmentations/multilayer_svd'
SINGLE_SEG_DIR= f'{DATA_DIR}/segmentations_single/multilayer_svd'

for d in [FEATURES_DIR, EIGS_DIR, SEGS_DIR, SINGLE_SEG_DIR]:
    os.makedirs(d, exist_ok=True)

print('Paths configured.')

## 4. Step 1 — Extract multi-layer Q+K features

Hooks into DINO ViT blocks 2, 5, 8, 11 (0-indexed = layers 3, 6, 9, 12).  
Computes `Q+K` per layer, concatenates → `(N, 4×384)` feature matrix per image.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}:', torch.cuda.get_device_name(i))

In [ ]:
from extract_multilayer_qk import extract_features_and_eigs

# Combined: extract Q+K features + SVD eigenvectors in one pass.
# Never writes feature files — saves ~37GB of disk vs the two-step approach.
extract_features_and_eigs(
    images_list    = f'{DATA_DIR}/lists/images.txt',
    images_root    = IMAGES_ROOT,
    model_name     = MODEL_NAME,
    batch_size     = BATCH_SIZE,
    eigs_output_dir = EIGS_DIR,
    K              = K,
    which_blocks   = '2,5,8,11',
)

## 5. Step 2 — Compute spectral eigenvectors via SVD

No N×N matrix ever built. Runs in O(N·D·K) via `torch.svd_lowrank`.

In [ ]:
# Step 2 (separate eigenvector extraction) is skipped —
# extract_features_and_eigs above already computed and saved eigenvectors.
print('Eigenvectors already computed in Step 1. Skipping.')

## 6. Step 3 — Segment images

K-means on eigenvectors (same downstream as baseline — eigenvector format is identical).

In [ ]:
# Multi-region segmentation (for object/semantic segmentation benchmarks)
import sys
sys.path.insert(0, f'{REPO}/extract')
from extract import extract_multi_region_segmentations, extract_single_region_segmentations

extract_multi_region_segmentations(
    features_dir            = FEATURES_DIR,
    eigs_dir                = EIGS_DIR,
    output_dir              = SEGS_DIR,
    adaptive                = False,
    non_adaptive_num_segments = NUM_SEGMENTS,
    infer_bg_index          = True,
    multiprocessing         = 0,
)

In [ ]:
# Single-region segmentation (for object localization / foreground-background)
extract_single_region_segmentations(
    features_dir = FEATURES_DIR,
    eigs_dir     = EIGS_DIR,
    output_dir   = SINGLE_SEG_DIR,
    threshold    = 0.0,
    multiprocessing = 0,
)

## 7. Visualize results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
from skimage.color import label2rgb
from pathlib import Path

COLORS = plt.cm.get_cmap('tab10').colors

def visualize_image(image_id, images_root, segs_dir, eigs_dir=None, n_cols=3):
    """Show original | segmentation overlay | eigenvectors."""
    img_path = Path(images_root) / f'{image_id}.jpg'
    seg_path = Path(segs_dir) / f'{image_id}.png'

    image = np.array(Image.open(img_path).convert('RGB'))
    segmap = np.array(Image.open(seg_path))
    segmap_full = cv2.resize(segmap, (image.shape[1], image.shape[0]),
                             interpolation=cv2.INTER_NEAREST)

    labels = np.unique(segmap)
    colors = [COLORS[i % len(COLORS)] for i in labels if i != 0]
    overlay = label2rgb(segmap_full, image=image,
                        colors=colors, bg_label=0, alpha=0.45)

    if eigs_dir is not None:
        eigs_path = Path(eigs_dir) / f'{image_id}.pth'
        eig_data = torch.load(eigs_path, map_location='cpu')
        eigvecs = eig_data['eigenvectors']  # (K, N)
        H_patch = segmap.shape[0]
        W_patch = segmap.shape[1]
        n_show = min(4, eigvecs.shape[0] - 1)
        n_cols = 2 + n_show

    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
    axes[0].imshow(image); axes[0].set_title('Image'); axes[0].axis('off')
    axes[1].imshow(overlay); axes[1].set_title('Segmentation'); axes[1].axis('off')

    if eigs_dir is not None:
        for j in range(n_show):
            ev = eigvecs[j + 1].numpy().reshape(H_patch, W_patch)
            axes[2 + j].imshow(ev, cmap='RdBu_r')
            axes[2 + j].set_title(f'Eigvec {j+1}')
            axes[2 + j].axis('off')

    plt.suptitle(image_id, fontsize=12)
    plt.tight_layout()
    plt.show()

print('Visualization function ready.')

In [ ]:
# Show a few sample images
seg_files = sorted(Path(SEGS_DIR).glob('*.png'))[:6]
for seg_file in seg_files:
    image_id = seg_file.stem
    visualize_image(image_id, IMAGES_ROOT, SEGS_DIR, EIGS_DIR)

## 8. Benchmark: Object Localization (CorLoc)

In [ ]:
# Build bounding boxes from single-region segmentations
from extract import extract_bboxes

BBOX_FILE = f'{DATA_DIR}/bboxes/multilayer_svd_e2_d3.pth'
os.makedirs(f'{DATA_DIR}/bboxes', exist_ok=True)

extract_bboxes(
    features_dir      = FEATURES_DIR,
    segmentations_dir = SINGLE_SEG_DIR,
    output_file       = BBOX_FILE,
    num_erode         = 2,
    num_dilate        = 3,
    skip_bg_index     = True,
)

In [ ]:
# CorLoc evaluation on VOC2012
sys.path.insert(0, f'{REPO}/object-localization')
import xml.etree.ElementTree as ET

def iou(box1, box2):
    """IoU of (xmin,ymin,xmax,ymax) boxes."""
    xi1 = max(box1[0], box2[0]); yi1 = max(box1[1], box2[1])
    xi2 = min(box1[2], box2[2]); yi2 = min(box1[3], box2[3])
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-6)

def get_gt_boxes(anno_path):
    tree = ET.parse(anno_path)
    root = tree.getroot()
    boxes = []
    for obj in root.findall('object'):
        bb = obj.find('bndbox')
        boxes.append([
            int(bb.find('xmin').text), int(bb.find('ymin').text),
            int(bb.find('xmax').text), int(bb.find('ymax').text),
        ])
    return boxes

bbox_list = torch.load(BBOX_FILE)
correct = 0
total = 0
for item in bbox_list:
    if not item['bboxes_original_resolution']:
        continue
    pred_box = item['bboxes_original_resolution'][0]  # take first box
    anno_path = f'{ANNO_ROOT}/{item["id"]}.xml'
    if not Path(anno_path).exists():
        continue
    gt_boxes = get_gt_boxes(anno_path)
    if not gt_boxes:
        continue
    best_iou = max(iou(pred_box, gt) for gt in gt_boxes)
    correct += int(best_iou >= 0.5)
    total += 1

corloc = 100.0 * correct / total if total > 0 else 0.0
print(f'CorLoc: {corloc:.1f}%  ({correct}/{total})')
print('Baseline (deep-spectral-segmentation): ~50-55%')

## 9. Benchmark: Object Segmentation (Jaccard)

In [ ]:
sys.path.insert(0, f'{REPO}/object-segmentation')

SEG_GT_DIR = SEG_OBJ_ROOT

if not Path(SEG_GT_DIR).exists():
    print('SegmentationObject GT not found — skipping object segmentation eval.')
else:
    from metrics import compute_jaccard
    jacc = compute_jaccard(
        pred_dir  = SINGLE_SEG_DIR,
        gt_dir    = SEG_GT_DIR,
        threshold = 0.5,
    )
    print(f'Jaccard: {jacc:.1f}%')
    print('Baseline (deep-spectral-segmentation): ~35-40%')

## 10. Benchmark: Semantic Segmentation (mIoU)

Requires the full semantic segmentation pipeline (bbox features → cluster → semantic map).

In [ ]:
from extract import (
    extract_bbox_features,
    extract_bbox_clusters,
    extract_semantic_segmentations,
)

BBOX_FEATURES_FILE = f'{DATA_DIR}/bboxes/bbox_features_multilayer.pth'
BBOX_CLUSTERS_FILE = f'{DATA_DIR}/bboxes/bbox_clusters_multilayer.pth'
SEM_SEGS_DIR       = f'{DATA_DIR}/semantic_segmentations/multilayer_svd'
os.makedirs(SEM_SEGS_DIR, exist_ok=True)

# 10a. Extract per-box features using DINO CLS token
extract_bbox_features(
    images_root = IMAGES_ROOT,
    bbox_file   = BBOX_FILE,
    model_name  = MODEL_NAME,
    output_file = BBOX_FEATURES_FILE,
)

# 10b. Cluster boxes into semantic categories (K=21 for VOC)
extract_bbox_clusters(
    bbox_features_file = BBOX_FEATURES_FILE,
    output_file        = BBOX_CLUSTERS_FILE,
    num_clusters       = 21,
    pca_dim            = 32,
    seed               = 0,
)

# 10c. Assign semantic labels to segmentation maps
extract_semantic_segmentations(
    segmentations_dir  = SEGS_DIR,
    bbox_clusters_file = BBOX_CLUSTERS_FILE,
    output_dir         = SEM_SEGS_DIR,
)

In [ ]:
# mIoU evaluation
sys.path.insert(0, f'{REPO}/semantic-segmentation')

SEG_CLASS_GT_DIR = SEG_ROOT
if not Path(SEG_CLASS_GT_DIR).exists():
    print('SegmentationClass GT not found — skipping semantic segmentation eval.')
else:
    from eval_utils import eval_predictions
    miou = eval_predictions(
        pred_dir = SEM_SEGS_DIR,
        gt_dir   = SEG_CLASS_GT_DIR,
        num_classes = 21,
    )
    print(f'mIoU: {miou:.1f}%')
    print('Baseline (deep-spectral-segmentation): ~15-18%')

## 11. Inspect eigenvectors for a single image

Useful for debugging / understanding what each eigenvector captures.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image as PILImage

# Pick any image
eig_files = sorted(Path(EIGS_DIR).glob('*.pth'))
sample_eig_file = eig_files[0]
sample_id = sample_eig_file.stem
print(f'Showing eigenvectors for: {sample_id}')

eig_data = torch.load(sample_eig_file, map_location='cpu')
feat_data = torch.load(Path(FEATURES_DIR) / f'{sample_id}.pth', map_location='cpu')

eigvecs = eig_data['eigenvectors']   # (K, N)
eigenvalues = eig_data['eigenvalues']  # (K,)
B, C, H, W = feat_data['shape']
P = feat_data['patch_size']
H_patch, W_patch = H // P, W // P

image = np.array(PILImage.open(f'{IMAGES_ROOT}/{sample_id}.jpg').convert('RGB'))

K_show = min(8, eigvecs.shape[0])
fig, axes = plt.subplots(2, K_show // 2, figsize=(3 * K_show // 2, 6))
axes = axes.flatten()
for i in range(K_show):
    ev = eigvecs[i].numpy().reshape(H_patch, W_patch)
    axes[i].imshow(ev, cmap='RdBu_r')
    axes[i].set_title(f'λ={eigenvalues[i]:.3f}')
    axes[i].axis('off')
plt.suptitle(f'Eigenvectors — {sample_id}', fontsize=13)
plt.tight_layout()
plt.show()

## 12. Compare with baseline

Run baseline pipeline for a direct comparison on the same images.

In [ ]:
# Baseline: extract K-only last-layer features + eigsh
from extract import extract_features as extract_features_baseline, extract_eigs

FEATURES_BASELINE = f'{DATA_DIR}/features/{MODEL_NAME}_baseline'
EIGS_BASELINE     = f'{DATA_DIR}/eigs/baseline_laplacian'
SEGS_BASELINE     = f'{DATA_DIR}/segmentations/baseline'

os.makedirs(FEATURES_BASELINE, exist_ok=True)

print('Extracting baseline K features (last layer only)...')
extract_features_baseline(
    images_list  = f'{DATA_DIR}/lists/images.txt',
    images_root  = IMAGES_ROOT,
    model_name   = MODEL_NAME,
    batch_size   = BATCH_SIZE,
    output_dir   = FEATURES_BASELINE,
    which_block  = -1,
)

print('Extracting baseline eigenvectors (eigsh on N×N Laplacian)...')
extract_eigs(
    images_root   = IMAGES_ROOT,
    features_dir  = FEATURES_BASELINE,
    output_dir    = EIGS_BASELINE,
    which_matrix  = 'laplacian',
    K             = K,
    image_color_lambda = 0,
    multiprocessing    = 0,
)

from extract import extract_multi_region_segmentations
extract_multi_region_segmentations(
    features_dir = FEATURES_BASELINE,
    eigs_dir     = EIGS_BASELINE,
    output_dir   = SEGS_BASELINE,
    non_adaptive_num_segments = NUM_SEGMENTS,
)

print('Baseline done.')

In [ ]:
# Side-by-side: baseline vs. multi-layer QK
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image as PILImage
from skimage.color import label2rgb

n_examples = 4
sample_files = sorted(Path(SEGS_DIR).glob('*.png'))[:n_examples]

fig, axes = plt.subplots(n_examples, 3, figsize=(12, 4 * n_examples))
if n_examples == 1:
    axes = [axes]

for row_idx, seg_file in enumerate(sample_files):
    image_id = seg_file.stem
    img_path = f'{IMAGES_ROOT}/{image_id}.jpg'
    baseline_seg_path = f'{SEGS_BASELINE}/{image_id}.png'
    new_seg_path      = str(seg_file)

    image = np.array(PILImage.open(img_path).convert('RGB'))
    H, W = image.shape[:2]

    def make_overlay(seg_path):
        seg = np.array(PILImage.open(seg_path))
        seg_full = cv2.resize(seg, (W, H), interpolation=cv2.INTER_NEAREST)
        labels = np.unique(seg_full)
        colors = [plt.cm.tab10.colors[i % 10] for i in labels if i != 0]
        return label2rgb(seg_full, image=image, colors=colors, bg_label=0, alpha=0.45)

    axes[row_idx][0].imshow(image)
    axes[row_idx][0].set_title('Original' if row_idx == 0 else '')
    axes[row_idx][0].axis('off')

    if Path(baseline_seg_path).exists():
        axes[row_idx][1].imshow(make_overlay(baseline_seg_path))
        axes[row_idx][1].set_title('Baseline (K, last layer)' if row_idx == 0 else '')
        axes[row_idx][1].axis('off')

    axes[row_idx][2].imshow(make_overlay(new_seg_path))
    axes[row_idx][2].set_title('Multi-Layer QK (ours)' if row_idx == 0 else '')
    axes[row_idx][2].axis('off')

plt.suptitle('Baseline vs. Multi-Layer QK Spectral Segmentation', fontsize=14)
plt.tight_layout()
plt.show()